# 05 — Comparativa de modelos

Cierra el **criterio de éxito E2 §2**: cuantificar si BERT mejora sobre TF-IDF+SVM y *en qué margen*. Reúne las métricas de test de los tres modelos (majority, TF-IDF+SVM, DistilBERT) en una tabla y un gráfico comparativos.

> Esta notebook **no necesita GPU**: solo lee los JSON de `reports/metrics/`. Ejecútala tras 02, 03 y 04.

## 1. Bootstrap

In [ ]:
REPO_URL = 'https://github.com/elvinsomon/pln-poc.git'

import os, sys, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir('/content/pln-poc'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/pln-poc'], check=True)
    os.chdir('/content/pln-poc')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from src.utils.colab import setup_environment
from src.utils.config import load_config

setup_environment(seed=42, project_root=PROJECT_ROOT)
cfg = load_config('base.yaml')
labels = cfg['classes']
print('cwd:', os.getcwd())

## 2. Cargar métricas de los tres modelos

> **Decisión:** `majority.json` es plano; `tfidf_svm.json` y `bert.json` tienen estructura `{val, test}`. Normalizamos todo al sub-dict de **test**.

In [ ]:
def load_test_metrics(name):
    path = Path(cfg['paths']['metrics']) / f'{name}.json'
    if not path.exists():
        raise FileNotFoundError(
            f"Falta {path}. Ejecuta la notebook correspondiente antes "
            f"(04 genera bert.json; 02/03 generan majority.json/tfidf_svm.json).")
    d = json.loads(path.read_text(encoding='utf-8'))
    return d['test'] if 'test' in d else d   # majority es plano

models = {'majority': 'Majority', 'tfidf_svm': 'TF-IDF+SVM', 'bert': 'DistilBERT'}
metrics = {k: load_test_metrics(k) for k in models}
print('cargados:', list(models.values()))

## 3. Tabla comparativa (TEST)

In [ ]:
rows = []
for key, nice in models.items():
    m = metrics[key]
    row = {'modelo': nice, 'accuracy': m['accuracy'], 'f1_macro': m['f1_macro']}
    for c in labels:
        row[f'f1_{c}'] = m['f1_per_class'][c]
    rows.append(row)
comparison_df = pd.DataFrame(rows).set_index('modelo').round(4)
comparison_df

## 4. Gráfico de barras agrupadas

In [ ]:
metric_cols = ['accuracy', 'f1_macro'] + [f'f1_{c}' for c in labels]
ax = comparison_df[metric_cols].T.plot(kind='bar', figsize=(10, 5))
ax.set_title('Comparativa en test')
ax.set_ylabel('score')
ax.set_ylim(0, 1)
ax.legend(title='modelo')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Margen de mejora BERT vs TF-IDF+SVM (criterio E2 §2)

In [ ]:
bert_m, svm_m = metrics['bert'], metrics['tfidf_svm']
delta_abs = bert_m['f1_macro'] - svm_m['f1_macro']
delta_rel = delta_abs / svm_m['f1_macro']
print(f"f1_macro  SVM : {svm_m['f1_macro']:.4f}")
print(f"f1_macro  BERT: {bert_m['f1_macro']:.4f}")
print(f"Δ absoluto : {delta_abs:+.4f}")
print(f"Δ relativo : {delta_rel:+.1%}")
print('\nΔ f1 por clase (BERT - SVM):')
for c in labels:
    print(f"  {c:9s}: {bert_m['f1_per_class'][c] - svm_m['f1_per_class'][c]:+.4f}")

## 6. Persistencia

In [ ]:
comp_path = Path(cfg['paths']['metrics']) / 'comparison.csv'
comparison_df.to_csv(comp_path)
print('saved:', comp_path)

## 7. Lectura

- Ambos modelos entrenados deben superar ampliamente el suelo `majority` (f1_macro 0.167), confirmando el **criterio E2 §1**.
- El margen BERT vs TF-IDF+SVM (celda anterior) responde al **criterio E2 §2**: si Δ es claramente positivo y consistente por clase, justifica el coste extra del modelo neuronal como *sistema objetivo* del TDT §7.2.
- Si el margen es marginal, la decisión de despliegue (TDT §7) debería ponderar interpretabilidad y latencia del SVM frente a la mejora de BERT.